In [ ]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dl = DataLoader(
    datasets.ImageFolder("data/Training", train_tf),
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=False
)

test_dl = DataLoader(
    datasets.ImageFolder("data/Testing", test_tf),
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=False
)

In [ ]:
model = nn.Sequential(
    nn.Conv2d(3, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.25),
    nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.25),
    nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.25),
    nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.25),
    nn.Flatten(),
    nn.Linear(256 * 14 * 14, 128), nn.ReLU(), nn.Dropout(0.1),
    nn.Linear(128, 4)
).to(device)

In [ ]:
opt = optim.Adam(model.parameters(), 1e-4)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
history = {'train_loss': []}
model.train()

for epoch in range(40):
    running_loss = 0

    for x, y in train_dl:
        opt.zero_grad()

        loss = loss_fn(model(x.to(device)), y.to(device))
        loss.backward()

        running_loss += loss.item()

        opt.step()

    avg_loss = running_loss / len(train_dl)
    history['train_loss'].append(avg_loss)
    print(f"Epoch {epoch + 1} Loss was {avg_loss:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Training Loss')
plt.title('Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model.eval()

test_loss, correct = 0.0, 0

with torch.no_grad():
    for x, y in test_dl:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        test_loss += loss_fn(logits, y).item() * y.size(0)

        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()

test_loss /= len(test_dl.dataset)
accuracy = 100 * correct / len(test_dl.dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", accuracy)

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in test_dl:
        x = x.to(device)
        logits = model(x)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=test_dl.dataset.classes, 
            yticklabels=test_dl.dataset.classes)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=test_dl.dataset.classes))

In [ ]:
import random
from torchvision.transforms.functional import to_pil_image

model.eval()
idx = random.randrange(len(test_dl.dataset))
img, label = test_dl.dataset[idx]

# Unnormalise with ImageNet stats for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
unnorm = (img * std + mean).clamp(0, 1)

plt.imshow(to_pil_image(unnorm))
plt.axis("off")
plt.title("Sample from Test Set")
plt.show()

with torch.no_grad():
    logits = model(img.unsqueeze(0).to(device))
    pred   = logits.argmax(dim=1).item()
    
class_names = test_dl.dataset.classes
print(f"Predicted:    {class_names[pred]}")
print(f"Ground Truth: {class_names[label]}")

In [ ]:
torch.save(model.state_dict(), "brain_tumor_model.pth")

In [ ]:
import joblib
# Save
joblib.dump(model, "brain_tumor_model.pkl")